# Ground-Truth Data Generation (LLM)

Generates user-style questions for both corpora (labels, regulations) with
an LLM, writing `data/ground-truth-labels.csv` and
`data/ground-truth-regulations.csv` (columns: `id, question`).

An offline template generator (`data/generate_ground_truth_offline.py`) ships
its output so retrieval eval runs with no API key. Re-run this to replace it
with LLM-generated questions. Requires `OPENAI_API_KEY`.

In [ ]:
import sys, json
sys.path.append("..")
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm
client = OpenAI()

In [ ]:
label_prompt = """
You emulate a drug-safety scientist using a label-search assistant.
Write 5 questions this user might ask that are answered by the label
section below. Use realistic phrasing INCLUDING lay synonyms for the
reaction (e.g. "granny flat"-style paraphrases such as "flesh-eating
infection" for "necrotizing fasciitis"), not just the exact label wording.

drug: {drug}
brand: {brand}
section: {section_name}
text: {text}

Output parsable JSON, no code blocks:
{{"questions": ["q1","q2","q3","q4","q5"]}}
""".strip()

In [ ]:
labels = pd.read_csv("../data/labels.csv")
labels = labels[~labels.version_note.str.startswith("pre-change")]

rows = []
for rec in tqdm(labels.to_dict("records")):
    raw = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content": label_prompt.format(**rec)}],
    ).choices[0].message.content
    for q in json.loads(raw)["questions"]:
        rows.append({"id": rec["id"], "question": q})

pd.DataFrame(rows).to_csv("../data/ground-truth-labels.csv", index=False)
len(rows)

Regulations: same pattern, prompting for questions about reporting duties, timelines, and supplement types.

In [ ]:
reg_prompt = """
You emulate a regulatory-affairs professional. Write 5 questions answered
by the regulation record below, using natural phrasing (not just quoting
the citation).

citation: {citation}
title: {title}
text: {text}

Output parsable JSON, no code blocks:
{{"questions": ["q1","q2","q3","q4","q5"]}}
""".strip()

regs = pd.read_csv("../data/regulations.csv")
rrows = []
for rec in tqdm(regs.to_dict("records")):
    raw = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content": reg_prompt.format(**rec)}],
    ).choices[0].message.content
    for q in json.loads(raw)["questions"]:
        rrows.append({"id": rec["id"], "question": q})

pd.DataFrame(rrows).to_csv("../data/ground-truth-regulations.csv", index=False)
len(rrows)